# 個人用 PLaMo アダプタを Colab の無料 GPU で作る

RAM 4GB の PC では学習できないため、学習だけ Colab（T4）で行う。所要は数十分（実測していない見込み）。

**前提**
- 上のメニュー「ランタイム → ランタイムのタイプを変更 → T4 GPU」。
- Hugging Face で `pfnet/plamo-3-translate-0.8b` の **PLaMo Community License に同意** し、読み取り用トークンを作っておく（左の鍵アイコン「シークレット」に `HF_TOKEN` として登録）。
- 学習に使う辞書は公開リポジトリのもの。私物の辞書を使う場合は、Google のサーバーに置くことになる点を確認してから。
- **できたアダプタは私用のみ**（PLaMo の派生モデル）。公開・共有しない。Colab 上のファイルはセッション終了で消える。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 https://github.com/ichirocc/MagiTranslator.git
%cd MagiTranslator
!pip -q install -r tools/llama/finetune/requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import snapshot_download
snapshot_download('pfnet/plamo-3-translate-0.8b', local_dir='plamo08-hf')

## 学習データ（アプリのプロンプトと同じ形、両方向）

In [ ]:
# testset.tsv: 評価に使う文（学習から除く）。リポジトリの翻訳テスト用文集の原文を使う
!python3 - <<'PY'
import csv
rows=[l.rstrip('\n').split('\t') for l in open('tools/eval/testsets/phrases141.tsv',encoding='utf-8') if l.strip() and not l.startswith('#')]
with open('testset.tsv','w',encoding='utf-8') as f:
    for r in rows[:200]:
        if len(r)>3: f.write(('ja' if r[1]=='ja-ne' else 'ne')+'\t'+r[3]+'\n')
PY
!python3 tools/llama/finetune/build_data.py dictionaries/phrases.tsv dictionaries/ja_ne_terms.tsv dictionaries/ne_ja_terms.tsv testset.tsv data/
!wc -l data/train.jsonl data/dev.tsv

## 学習（約1.1エポック。PC で最良だった step 600 前後）

In [ ]:
!python3 tools/llama/finetune/train_lora.py plamo08-hf data/train.jsonl lora/ 2 600

## アプリ用の GGUF アダプタに変換して保存

In [ ]:
!sh tools/llama/finetune/export_adapter.sh plamo08-hf lora/step600 personal_lora.gguf
!ls -la personal_lora.gguf
from google.colab import files
files.download('personal_lora.gguf')

スマホでは 設定 →「辞書を読み込む」で `personal_lora.gguf` を選ぶ（辞書と同じボタン）。ベースモデルは変わらない。

効果の確認は PC で `tools/llama/finetune/README.md` の「Check an adapter」の手順（chrF と話題外れの件数）。